# Gold: patient_360

Builds `adb_caresync.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance_providers and aggregated lab_results, per STTM `sttm_silver_to_gold.xlsx` v1.0. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
-- Grain: one row per patient. Driving table: silver.patients (LEFT JOINs only; no rows dropped)
-- Note: silver.lab_results contains exact duplicate lab_result_id rows; dedup to one row per lab test before aggregating
CREATE OR REPLACE TABLE adb_caresync.gold.patient_360
COMMENT 'Gold patient 360: one row per patient with hospital, insurance and lab test summary'
AS
WITH lab_results_dedup AS (
  SELECT *
  FROM adb_caresync.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC, insert_timestamp DESC) = 1
),
lab_agg AS (
  SELECT
    patient_id,
    count(*)                             AS total_tests,
    count_if(result_status = 'Abnormal') AS abnormal_test_count,
    count_if(result_status = 'Critical') AS critical_test_count,
    max(test_date)                       AS last_test_date
  FROM lab_results_dedup
  GROUP BY patient_id
)
SELECT
  p.patient_id                                                                 AS patient_id,
  p.first_name                                                                 AS first_name,
  p.last_name                                                                  AS last_name,
  CAST(p.date_of_birth AS DATE)                                                AS date_of_birth,
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender                                                                     AS gender,
  p.city                                                                       AS city,
  p.state                                                                      AS state,
  p.registered_hospital_id                                                     AS hospital_id,
  h.hospital_name                                                              AS hospital_name,
  p.insurance_provider_id                                                      AS insurance_provider_id,
  ip.provider_name                                                             AS insurance_provider_name,
  CAST(coalesce(la.total_tests, 0) AS INT)                                     AS total_tests,
  CAST(coalesce(la.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
  CAST(coalesce(la.critical_test_count, 0) AS INT)                             AS critical_test_count,
  la.last_test_date                                                            AS last_test_date
FROM adb_caresync.silver.patients p
LEFT JOIN adb_caresync.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg la
  ON p.patient_id = la.patient_id

In [0]:
# Return the record count of the rebuilt gold table to the caller (e.g. job / dbutils.notebook.run)
updated_df = spark.table("adb_caresync.gold.patient_360")
record_count = updated_df.count()
print(f"adb_caresync.gold.patient_360 record count: {record_count}")
dbutils.notebook.exit(str(record_count))